# Problem 419 - Look and Say Sequence

The <strong>look and say</strong> sequence goes 1, 11, 21, 1211, 111221, 312211, 13112221, 1113213211, ...
The sequence starts with 1 and all other members are obtained by describing the previous member in terms of consecutive digits.
It helps to do this out loud:
1 is 'one one' → 11
11 is 'two ones' → 21
21 is 'one two and one one' → 1211 
1211 is 'one one, one two and two ones' → 111221
111221 is 'three ones, two twos and one one' → 312211
...


Define $A(n)$, $B(n)$ and $C(n)$ as the number of ones, twos and threes in the $n$'th element of the sequence respectively.
One can verify that $A(40) = 31254$, $B(40) = 20259$ and $C(40) = 11625$.


Find $A(n)$, $B(n)$ and $C(n)$ for $n = 10^{12}$.
Give your answer modulo $2^{30}$ and separate your values for $A$, $B$ and $C$ by a comma.
E.g. for $n = 40$ the answer would be 31254,20259,11625

## Solution.

https://sites.math.rutgers.edu/~zeilberg/EM12/ConwayWW.pdf

https://njohnston.ca/2010/10/a-derivation-of-conways-degree-71-look-and-say-polynomial/

http://www.se16.info/js/lands2.htm

In [166]:
from functools import cache

Get atoms of Conway's sequence and their decompositions after evvolution.

In [174]:
import numpy as np
 
atoms = ["1112",
    "1112133",
    "111213322112",
    "111213322113",
    "1113",
    "11131",
    "111311222112",
    "111312",
    "11131221",
    "1113122112",
    "1113122113",
    "11131221131112",
    "111312211312",
    "11131221131211",
    "111312211312113211",
    "111312211312113221133211322112211213322112",
    "111312211312113221133211322112211213322113",
    "11131221131211322113322112",
    "11131221133112",
    "1113122113322113111221131221",
    "11131221222112",
    "111312212221121123222112",
    "111312212221121123222113",
    "11132",
    "1113222",
    "1113222112",
    "1113222113",
    "11133112",
    "12",
    "123222112",
    "123222113",
    "12322211331222113112211",
    "13",
    "131112",
    "13112221133211322112211213322112",
    "13112221133211322112211213322113",
    "13122112",
    "132",
    "13211",
    "132112",
    "1321122112",
    "132112211213322112",
    "132112211213322113",
    "132113",
    "1321131112",
    "13211312",
    "1321132",
    "13211321",
    "132113212221",
    "13211321222113222112",
    "1321132122211322212221121123222112",
    "1321132122211322212221121123222113",
    "13211322211312113211",
    "1321133112",
    "1322112",
    "1322113",
    "13221133112",
    "1322113312211",
    "132211331222113112211",
    "13221133122211332",
    "22",
    "3",
    "3112",
    "3112112",
    "31121123222112",
    "31121123222113",
    "3112221",
    "3113",
    "311311",
    "31131112",
    "3113112211",
    "3113112211322112",
    "3113112211322112211213322112",
    "3113112211322112211213322113",
    "311311222",
    "311311222112",
    "311311222113",
    "3113112221131112",
    "311311222113111221",
    "311311222113111221131221",
    "31131122211311122113222",
    "3113112221133112",
    "311312",
    "31132",
    "311322113212221",
    "311332",
    "3113322112",
    "3113322113",
    "312",
    "312211322212221121123222113",
    "312211322212221121123222112",
    "32112"]
 
N = len(atoms)
idx = {a: i for i, a in enumerate(atoms)}
 
official = {
    'H':  ('22', ['H']),
    'He': ('13112221133211322112211213322112', ['Hf', 'Pa', 'H', 'Ca', 'Li']),
    'Li': ('312211322212221121123222112', ['He']),
    'Be': ('111312211312113221133211322112211213322112', ['Ge', 'Ca', 'Li']),
    'B':  ('1321132122211322212221121123222112', ['Be']),
    'C':  ('3113112211322112211213322112', ['B']),
    'N':  ('111312212221121123222112', ['C']),
    'O':  ('132112211213322112', ['N']),
    'F':  ('31121123222112', ['O']),
    'Ne': ('111213322112', ['F']),
    'Na': ('123222112', ['Ne']),
    'Mg': ('3113322112', ['Pm', 'Na']),
    'Al': ('1113222112', ['Mg']),
    'Si': ('1322112', ['Al']),
    'P':  ('311311222112', ['Ho', 'Si']),
    'S':  ('1113122112', ['P']),
    'Cl': ('132112', ['S']),
    'Ar': ('3112', ['Cl']),
    'K':  ('1112', ['Ar']),
    'Ca': ('12', ['K']),
    'Sc': ('3113112221133112', ['Ho', 'Pa', 'H', 'Ca', 'Co']),
    'Ti': ('11131221131112', ['Sc']),
    'V':  ('13211312', ['Ti']),
    'Cr': ('31132', ['V']),
    'Mn': ('111311222112', ['Cr', 'Si']),
    'Fe': ('13122112', ['Mn']),
    'Co': ('32112', ['Fe']),
    'Ni': ('11133112', ['Zn', 'Co']),
    'Cu': ('131112', ['Ni']),
    'Zn': ('312', ['Cu']),
    'Ga': ('13221133122211332', ['Eu', 'Ca', 'Ac', 'H', 'Ca', 'Zn']),
    'Ge': ('31131122211311122113222', ['Ho', 'Ga']),
    'As': ('11131221131211322113322112', ['Ge', 'Na']),
    'Se': ('13211321222113222112', ['As']),
    'Br': ('3113112211322112', ['Se']),
    'Kr': ('11131221222112', ['Br']),
    'Rb': ('1321122112', ['Kr']),
    'Sr': ('3112112', ['Rb']),
    'Y':  ('1112133', ['Sr', 'U']),
    'Zr': ('12322211331222113112211', ['Y', 'H', 'Ca', 'Tc']),
    'Nb': ('1113122113322113111221131221', ['Er', 'Zr']),
    'Mo': ('13211322211312113211', ['Nb']),
    'Tc': ('311322113212221', ['Mo']),
    'Ru': ('132211331222113112211', ['Eu', 'Ca', 'Tc']),
    'Rh': ('311311222113111221131221', ['Ho', 'Ru']),
    'Pd': ('111312211312113211', ['Rh']),
    'Ag': ('132113212221', ['Pd']),
    'Cd': ('3113112211', ['Ag']),
    'In': ('11131221', ['Cd']),
    'Sn': ('13211', ['In']),
    'Sb': ('3112221', ['Pm', 'Sn']),
    'Te': ('1322113312211', ['Eu', 'Ca', 'Sb']),
    'I':  ('311311222113111221', ['Ho', 'Te']),
    'Xe': ('11131221131211', ['I']),
    'Cs': ('13211321', ['Xe']),
    'Ba': ('311311', ['Cs']),
    'La': ('11131', ['Ba']),
    'Ce': ('1321133112', ['La', 'H', 'Ca', 'Co']),
    'Pr': ('31131112', ['Ce']),
    'Nd': ('111312', ['Pr']),
    'Pm': ('132', ['Nd']),
    'Sm': ('311332', ['Pm', 'Ca', 'Zn']),
    'Eu': ('1113222', ['Sm']),
    'Gd': ('13221133112', ['Eu', 'Ca', 'Co']),
    'Tb': ('3113112221131112', ['Ho', 'Gd']),
    'Dy': ('111312211312', ['Tb']),
    'Ho': ('1321132', ['Dy']),
    'Er': ('311311222', ['Ho', 'Pm']),
    'Tm': ('11131221133112', ['Er', 'Ca', 'Co']),
    'Yb': ('1321131112', ['Tm']),
    'Lu': ('311312', ['Yb']),
    'Hf': ('11132', ['Lu']),
    'Ta': ('13112221133211322112211213322113', ['Hf', 'Pa', 'H', 'Ca', 'W']),
    'W':  ('312211322212221121123222113', ['Ta']),
    'Re': ('111312211312113221133211322112211213322113', ['Ge', 'Ca', 'W']),
    'Os': ('1321132122211322212221121123222113', ['Re']),
    'Ir': ('3113112211322112211213322113', ['Os']),
    'Pt': ('111312212221121123222113', ['Ir']),
    'Au': ('132112211213322113', ['Pt']),
    'Hg': ('31121123222113', ['Au']),
    'Tl': ('111213322113', ['Hg']),
    'Pb': ('123222113', ['Tl']),
    'Bi': ('3113322113', ['Pm', 'Pb']),
    'Po': ('1113222113', ['Bi']),
    'At': ('1322113', ['Po']),
    'Rn': ('311311222113', ['Ho', 'At']),
    'Fr': ('1113122113', ['Rn']),
    'Ra': ('132113', ['Fr']),
    'Ac': ('3113', ['Ra']),
    'Th': ('1113', ['Ac']),
    'Pa': ('13', ['Th']),
    'U':  ('3', ['Pa']),
}


Construct transition matrix $T$. $(i,j)$ entry is the $(\text{number of times i-th atom appears after evolving j-th atom})$.

We can represent each number in the sequence (after some term) as $1\times n$ vector $v$ where $i$-th entry is the number of times atom $i$ appears. Then $T^n v$ will represent $v$ after $n$ evolutions.

In [175]:
name_to_string = {name: s for name, (s, succ) in official.items()}

T = np.zeros((N, N), dtype=object)     
for name, (s, succ) in official.items():
    j = idx[s]
    for succ_name in succ:
        i = idx[name_to_string[succ_name]]
        T[i, j] += 1

In [181]:
def solv(n = 10**12, MOD = 2**30):
    i = 1
    s = '1'

    while decompose(s) is None:
        i += 1
        s = evolution(s)

    EXP = n - i
    P = matrix_power_mod(T, EXP, MOD)
    v = P @ decompose(s).T

    A, B, C  = 0, 0,0 

    for i, el in enumerate(v):
        A += atoms[i].count('1') * el
        B += atoms[i].count('2') * el
        C += atoms[i].count('3') * el

    return (f"{int(A) % MOD},{int(B) % MOD},{int(C) % MOD}")


In [182]:
solv()

'998567458,1046245404,43363922'

In [183]:
solv(n=40) == "31254,20259,11625"

True

--------

Some things I used to for experimentations but never ended up in the main solution.

The biggest caveat was to understand that the decomposition of atoms after evolution is non-unique but there exists one (or at least one) good one that indeed satisfies atom defintion in the sense that they will create "perpendicular" sequences not interacting with each other. I have not derived that part myself but copied from an online source. It would be interestening to investigate how to obtain "atoms" and how to find that they behave appropriately.

In [160]:
@cache
def evolution(prev):
    i = 0
    new = ''
    while i < len(prev):
        j = i
        while prev[i] == prev[j]:                        
            if j == len(prev)-1:
                new += str(j-i+1) 
                new += prev[i]
                return new
            j += 1
                    
        new += str(j-i) 
        new += prev[i]
        i = j


def decompose(s):
    if s in atoms:
        ans = [0] * N
        idx = atoms.index(s)
        ans[idx] = 1
        return np.array(ans)

    for atom in atoms:
        if s.startswith(atom):
            s_new = s[len(atom):]

            if decompose(s_new) is not None:
                ans = [0] * N
                ans[atoms.index(atom)] = 1
                ans = np.array(ans)
                return ans + decompose(s_new)          

    return None

def matrix_power_mod(T, exp, mod):
    T = np.asarray(T, dtype=np.int64) % mod
    n = T.shape[0]
    result = np.eye(n, dtype=np.int64)

    while exp > 0:
        if exp & 1:
            result = (result @ T) % mod
        T = (T @ T) % mod
        exp >>= 1

    return result